<table align="left"><tr><td>
<a href="https://colab.research.google.com/github/kikim6114/nlp2026/blob/main/python_tutorial/00-9.PyTorch_Tutorial2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="코랩에서 실행하기"/></a>
</td></tr></table>

### 데모: Word Window Classification: 단어 단위 문맥 분류

문장 속 하나의 단어만을 보는 것으로는 부족하기 때문에,  
**단어 주변의 문맥**(context)을 함께 고려해야 한다.  
이를 위해 **Word Window Classification**이라는 고전적인 기법을 사용하여  
**단어 단위로 위치 정보**(Location)를 분류하는 문제를 해결한다.

---

### 문제 정의

- **목표**: 문장의 각 단어가 `LOCATION`인지 아닌지를 판단하는 모델 구축
- **가정**: `LOCATION`은 항상 **한 단어짜리**만 존재 (예: `Paris`는 OK, `San Francisco`는 X)

---

### 데이터 만들기

- 각 단어에 대해 `"앞 단어, 현재 단어, 뒤 단어"`로 이루어진 **윈도우(window)** 생성
- 각 윈도우는 **하나의 입력**으로 사용되며,  
  **중앙 단어의 레이블**(LOCATION 여부)을 예측

---

### 모델링

- **입력**: 윈도우 내 단어들의 벡터(임베딩 또는 원-핫)
- **출력**: 중앙 단어가 `LOCATION`인지 아닌지를 판별하는 **이진 분류(binary classification)**

---

### 훈련

- **손실 함수**:
  - `nn.BCELoss()` 또는 `nn.CrossEntropyLoss()`  
- **옵티마이저**:
  - `Adam`, `SGD` 등

---

### 예측

- 문장을 입력으로 받아서  
  각 단어가 **LOCATION**인지 여부를 **순서대로 판단**


### Data

모든 머신 러닝 프로젝트의 첫 번째 작업은 훈련 세트를 설정하는 것이다. 일반적으로 사용할 훈련 코퍼스가 있을 것이다. NLP 작업에서 코퍼스는 일반적으로 각 행이 문장이나 테이블 형식 데이터 포인트에 해당하는 `.txt` 또는 `.csv` 파일이다. 이 간단한 작업에서는 데이터와 해당 레이블을 이미 `Python` 리스트로 읽었다고 가정하겠다.

In [2]:
# 문장들로 구성된 원시 텍스트 데이터 (corpus)
# 자연어 처리에서 학습이나 분석의 입력 데이터로 사용됨
corpus = [
          "We always come to Paris",
          "The professor is from Australia",
          "I live in Stanford",
          "He comes from Taiwan",
          "The capital of Turkey is Ankara"
         ]

### 전처리(Preprocessing)의 중요성

분석의 목적에 따라 어떤 전처리 과정을 수행할지가 결정되므로,  
**전처리 과정은 자연어 처리에서 필수적인 단계**이다.

---

### 주요 전처리 단계

1. **Tokenization (토큰화)**  
   - 문장을 **단어 단위로 분리**  
   - 예: `"I love Paris"` → `["I", "love", "Paris"]`

2. **Lowercasing (소문자 변환)**  
   - 대소문자 구분 없이 처리하기 위해 **모든 단어를 소문자**로 변환  
   - 예: `"Paris"` → `"paris"`

3. **Noise removal (노이즈 제거)**  
   - 모델 학습에 방해되는 **특수문자, 기호, 이모지** 등을 제거  
   - 예: `"hello! 😊"` → `"hello"`

4. **Stop words removal (불용어 제거)**  
   - 자주 등장하지만 **정보량이 적은 단어들** 제거  
   - 예: `a`, `the`, `is`, `in`, `on`, ...

---

이러한 전처리 과정을 통해 **데이터의 품질을 높이고, 모델 성능 향상**에 기여할 수 있다.


In [3]:
# 훈련 예제를 생성하기 위해 사용할 전처리 함수
# 이 함수는 간단하게 글자를 소문자로 바꾸고 단어를 토큰화한다.
def preprocess_sentence(sentence):
  return sentence.lower().split() #띄어쓰기 기준으로 구분

# 학습용 문장 데이터 생성
# 각 문장에 전처리 함수(preprocess_sentence)를 적용하여
# 모델이 사용할 수 있는 형태로 변환
train_sentences = [preprocess_sentence(sent) for sent in corpus]

train_sentences

[['we', 'always', 'come', 'to', 'paris'],
 ['the', 'professor', 'is', 'from', 'australia'],
 ['i', 'live', 'in', 'stanford'],
 ['he', 'comes', 'from', 'taiwan'],
 ['the', 'capital', 'of', 'turkey', 'is', 'ankara']]

각 훈련 예제마다 해당 레이블도 있어야 한다. 모델의 목표가 어떤 단어가 `LOCATION`에 해당하는지 결정하는 것임을 상기하라. 즉, 모델이 `LOCATION`이 아닌 모든 단어에 대해 `0`을 출력하고, `LOCATION`인 단어에 대해 `1`을 출력하기를 원한다.

In [4]:
# 말뭉치(corpus)에 등장하는 지명(location) 단어 집합 정의
locations = set(["australia", "ankara", "paris", "stanford", "taiwan", "turkey"])

# 각 문장의 단어가 지명이면 1, 아니면 0으로 라벨 생성
# → 토큰 단위 분류를 위한 학습용 정답(label) 데이터 생성
train_labels = [[1 if word in locations else 0 for word in sent]
                for sent in train_sentences]

train_labels

[[0, 0, 0, 0, 1],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 1],
 [0, 0, 0, 1],
 [0, 0, 0, 1, 0, 1]]

### Converting Words to Embeddings (단어를 임베딩으로 변환)

우리의 입력은 `"paris"`, `"live"`처럼 **문자열 형태의 단어**이지만,  
머신러닝/딥러닝 모델은 **숫자 벡터**로만 연산이 가능하기 때문에  
**문자열을 숫자로 변환**하는 과정이 필요한다.

---

임베딩 조회 테이블 `E`가 있다고 상상해 보자. 여기서 각 행은 임베딩에 해당한다. 즉, 어휘에 있는 각 단어는 이 테이블에서 해당하는 임베딩 행 `i`를 갖는다. 단어의 임베딩을 찾으려면 다음 단계를 따른다:
1. 임베딩 테이블에서 단어의 해당 인덱스 `i`를 찾는다: `word->index`.
2. 임베딩 테이블에 인덱싱하여 임베딩을 얻는다: `index->embedding`.

첫 번째 단계를 살펴보자. 어휘에 있는 모든 단어에 해당 인덱스를 지정해야 한다. 다음과 같이 할 수 있다:
1. 코퍼스에서 모든 고유 단어를 찾는다.
2. 각 단어에 인덱스를 지정한다.

In [6]:
# 말뭉치에 등장하는 모든 고유 단어(어휘집, vocabulary) 생성
# → 모델이 인식할 수 있는 단어 목록
vocabulary = set(w for s in train_sentences for w in s)

vocabulary

{'always',
 'ankara',
 'australia',
 'capital',
 'come',
 'comes',
 'from',
 'he',
 'i',
 'in',
 'is',
 'live',
 'of',
 'paris',
 'professor',
 'stanford',
 'taiwan',
 'the',
 'to',
 'turkey',
 'we'}

### Out-of-Vocabulary (OOV) 단어 처리

현재의 **vocabulary**(어휘 사전)는 우리가 가진 말뭉치(corpus)에 등장한 모든 단어들을 포함하고 있다.  
하지만 **테스트 시점**(test time)에는 **어휘 사전에 없는 단어들**(out-of-vocabulary, OOV)을 마주칠 수 있다.

---

### 왜 OOV 처리가 필요한가?

모델은 예측 시에 **중심 단어 뿐만 아니라 주변 단어들의 문맥**도 함께 보기 때문에, **알 수 없는 단어라도 그 주변 문맥을 통해 의미를 추론**할 수 있다.

→ 따라서, OOV 단어를 완전히 배제하지 않고, 대신 **특수 토큰** `<unk>`(unknown)으로 대체해 처리한다.

---

### `<unk>` 토큰의 역할

- 어휘 사전에 없는 모든 단어는 `<unk>` 토큰으로 대체됨
- 이 토큰은 **고유하게만 설정**되어 있다면 이름은 꼭 `<unk>`일 필요는 없음  
  - 예: `<unknown>`, `[UNK]`, `@@unk@@` 등

---

이렇게 하면 **모델이 학습되지 않은 단어**가 등장하더라도 일관된 방식으로 처리할 수 있으며, 문맥 기반으로 충분히 예측이 가능해진다.


In [ ]:
# Add the unknown token to our vocabulary
vocabulary.add("<unk>")

### 왜 Word Window Classification인가?

우리는 앞서 이 태스크가 **Word Window Classification**이라고 불리는 이유에 대해 언급한 적이 있다.  
그 이유는, **모델이 단어를 예측할 때 해당 단어뿐만 아니라 그 주변 단어들까지 함께 고려하기 때문**이다.

---

### 예시 문장: `"We always come to Paris"`

- 정답 레이블: `[0, 0, 0, 0, 1]`  
- 이유: 오직 마지막 단어 `"Paris"`만 **LOCATION**(장소명)이기 때문

---

### 단어 하나만 보면 안 되는 이유

모델이 `"Paris"` 하나만 보고 예측하게 된다면,  
예를 들어 `"to"`라는 단어가 자주 LOCATION 앞에 등장한다는  
**중요한 문맥 정보**를 활용할 수 없다.

→ **그래서 Word Window라는 개념을 도입**한다.

---

### Word Window 개념

- 예측 시, 중심 단어 기준으로 **앞뒤 N개의 단어**(**총 2N + 1개**)를 함께 고려
- 윈도우 크기(`window size`)가 1이라면:
  - 입력 단어 수: 앞 1, 중심 1, 뒤 1 → **총 3개의 단어**

#### 예시 (윈도우 크기 = 1):
중심 단어 `"Paris"` → 입력: `["to", "Paris", <pad>]`  
(※ `"Paris"`는 문장의 마지막 단어이므로 뒤 단어가 없음)

---

### 고정된 입력 크기의 문제

PyTorch 모델을 초기화할 때는 입력 크기를 **미리 고정**해야 한다.  
하지만 문장의 처음이나 끝에 위치한 단어들은 **윈도우의 일부가 부족**할 수 있다.

→ 예: `"Paris"`는 마지막 단어이므로 뒤 단어가 없음  
→ 예: `"We"`는 첫 단어이므로 앞 단어가 없음

---

### 해결책: `<pad>` 토큰

- 문장의 앞과 뒤에 `<pad>` 같은 **특수 토큰**을 추가
- 모든 단어가 항상 **고정된 윈도우 크기**를 유지할 수 있도록 보장

이렇게 하면 모델은 **항상 동일한 입력 크기**로 데이터를 받아 처리할 수 있다.


In [ ]:
# 패딩에 사용할 특수 토큰을 vocabulary에 추가
vocabulary.add("<pad>")

# 문장의 앞뒤에 pad 토큰을 추가하는 함수
# → 윈도우 기반 모델에서 문장 경계 처리에 사용됨
def pad_window(sentence, window_size, pad_token="<pad>"):
    window = [pad_token] * window_size
    return window + sentence + window

# 패딩 적용 예시 (문장 앞뒤에 pad 토큰 2개씩 추가)
window_size = 2
pad_window(train_sentences[0], window_size=window_size)

['<pad>', '<pad>', 'we', 'always', 'come', 'to', 'paris', '<pad>', '<pad>']

Now that our vocabularly is ready, let's assign an index to each of our words.

In [ ]:
# vocabulary를 리스트로 변환하여 인덱싱 가능하도록 함
# 정렬은 필수는 아니지만, 단어-인덱스 매핑을 보기 좋게 하기 위해 수행
# 또한 <pad> 토큰의 인덱스를 0으로 두면 이후 패딩 처리에서 편리함
ix_to_word = sorted(list(vocabulary))

# 각 단어에 고유한 정수 인덱스를 부여하는 딕셔너리 생성
# → 단어를 모델 입력용 숫자로 변환하기 위한 매핑
word_to_ix = {word: ind for ind, word in enumerate(ix_to_word)}

word_to_ix

{'<pad>': 0,
 '<unk>': 1,
 'always': 2,
 'ankara': 3,
 'australia': 4,
 'capital': 5,
 'come': 6,
 'comes': 7,
 'from': 8,
 'he': 9,
 'i': 10,
 'in': 11,
 'is': 12,
 'live': 13,
 'of': 14,
 'paris': 15,
 'professor': 16,
 'stanford': 17,
 'taiwan': 18,
 'the': 19,
 'to': 20,
 'turkey': 21,
 'we': 22}

In [ ]:
ix_to_word[1]

'<unk>'

Great! We are ready to convert our training sentences into a sequence of indices corresponding to each token.

In [ ]:
# 문장의 각 토큰을 해당하는 정수 인덱스로 변환하는 함수
# vocabulary에 없는 단어는 <unk> 토큰의 인덱스로 처리
def convert_token_to_indices(sentence, word_to_ix):
    indices = []
    for token in sentence:
        if token in word_to_ix:
            index = word_to_ix[token]
        else:
            index = word_to_ix["<unk>"]   # 어휘집에 없는 단어 처리
        indices.append(index)
    return indices

# 위 함수의 간결한 구현 (dictionary get 활용)
def _convert_token_to_indices(sentence, word_to_ix):
    return [word_to_ix.get(token, word_to_ix["<unk>"]) for token in sentence]

# 예시 문장을 정수 인덱스로 변환한 뒤 다시 단어로 복원
example_sentence = ["we", "always", "come", "to", "kuwait"]
example_indices = convert_token_to_indices(example_sentence, word_to_ix)
restored_example = [ix_to_word[ind] for ind in example_indices]

print(f"Original sentence is: {example_sentence}")
print(f"Going from words to indices: {example_indices}")
print(f"Going from indices to words: {restored_example}")

Original sentence is: ['we', 'always', 'come', 'to', 'kuwait']
Going from words to indices: [22, 2, 6, 20, 1]
Going from indices to words: ['we', 'always', 'come', 'to', '<unk>']


In the example above, `kuwait` shows up as `<unk>`, because it is not included in our vocabulary. Let's convert our `train_sentences` to `example_padded_indices`.

In [ ]:
# 학습용 문장들을 모두 정수 인덱스 시퀀스로 변환
# → 모델 입력을 위한 수치 데이터 형태로 변환
example_padded_indices = [convert_token_to_indices(s, word_to_ix)
                          for s in train_sentences]

example_padded_indices

[[22, 2, 6, 20, 15],
 [19, 16, 12, 8, 4],
 [10, 13, 11, 17],
 [9, 7, 8, 18],
 [19, 5, 14, 21, 12, 3]]

### 임베딩

- 이제 각 단어에 대한 인덱스가 있으므로, PyTorch의 `nn.Embedding` 클래스를 사용하여 임베딩 테이블을 만들 수 있다.
- 이는 `nn.Embedding(num_words, embedding_dimension)`로 호출되며, 여기서 `num_words`는 어휘의 단어 수이고 `embedding_dimension`은 원하는 임베딩 차원이다.
- `nn.Embedding`에는 특별한 점이 없다: 이는 단지 학습 가능한 `N x E` 차원 텐서를 감싸는 래퍼 클래스(wrapper class)일 뿐이다.
- 여기서 `N`은 어휘의 단어 수이고 `E`는 임베딩 차원 수이다.
- 이 테이블은 처음에는 무작위이지만, 시간이 지나면서 변할 것이다.
- 네트워크를 훈련시키면서 기울기가 임베딩 레이어까지 역전파되어 단어 임베딩이 업데이트된다.
- 모델에서 사용할 임베딩 레이어를 모델에서 초기화할 것이지만, 여기서 예제를 보여준다.

In [ ]:
# 단어 임베딩 테이블 생성
# → vocabulary 크기 × 임베딩 차원 크기의 행렬이 만들어짐
embedding_dim = 5
embeds = nn.Embedding(len(vocabulary), embedding_dim)

# 임베딩 테이블의 학습 파라미터(가중치 행렬) 확인
list(embeds.parameters())

[Parameter containing:
 tensor([[-0.8704, -0.2992,  0.5500,  0.0036, -0.3180],
         [-0.7567, -1.8032, -0.3749,  1.6157, -0.4348],
         [-0.1822,  0.0983,  1.3873,  1.9386, -1.3971],
         [-0.1304, -0.7972, -0.8293, -0.4809,  0.8129],
         [-0.1279, -0.0751,  1.7489,  0.5025, -1.6097],
         [-1.2070,  1.1461,  0.9074,  0.0253, -0.0606],
         [ 1.2347,  1.5978,  1.7446,  0.9985,  0.4297],
         [-0.8560,  0.8482,  0.0927, -0.8030, -0.9074],
         [-1.1697,  0.6581, -0.8054,  0.7381, -0.1294],
         [ 0.1687,  0.5908, -0.5600,  0.6541,  0.3468],
         [ 0.9635,  0.5785, -2.3571,  0.2742, -0.7405],
         [ 0.3447, -2.6803, -0.6071, -0.1295,  1.2861],
         [ 0.6089, -0.0494, -1.0986,  0.6936,  1.4630],
         [-1.4969, -0.3686,  0.6553,  0.5097,  0.1691],
         [-0.4210, -0.5844,  0.2633, -0.0262,  1.7294],
         [ 1.3983, -0.6694, -0.3386, -0.3865,  0.0711],
         [ 1.2956,  2.3043, -1.4080,  0.1569, -0.8630],
         [-1.4318,  0.150

어휘에 있는 단어의 임베딩을 얻으려면 조회 텐서를 만들기만 하면 된다. 조회 텐서는 우리가 조회하려는 인덱스를 포함하는 텐서일 뿐이다. `nn.Embedding` 클래스는 Long Tensor 타입의 인덱스 텐서를 기대하므로, 이에 맞춰 텐서를 만들어야 한다.

In [ ]:
# 단어 "paris"의 임베딩 벡터 조회

# 단어를 정수 인덱스로 변환
index = word_to_ix["paris"]

# 임베딩 레이어 입력을 위해 정수 인덱스를 Long 타입 텐서로 변환
index_tensor = torch.tensor(index, dtype=torch.long)

# 해당 인덱스에 대응하는 임베딩 벡터 추출
paris_embed = embeds(index_tensor)

paris_embed

tensor([ 1.3983, -0.6694, -0.3386, -0.3865,  0.0711],
       grad_fn=<EmbeddingBackward0>)

In [ ]:
# 여러 단어의 임베딩 벡터를 동시에 조회

# 각 단어를 정수 인덱스로 변환
index_paris = word_to_ix["paris"]
index_ankara = word_to_ix["ankara"]

# 인덱스들을 하나의 리스트로 구성
indices = [index_paris, index_ankara]

# 임베딩 레이어 입력을 위해 Long 타입 텐서로 변환
indices_tensor = torch.tensor(indices, dtype=torch.long)

# 여러 인덱스에 대응하는 임베딩 벡터들을 한 번에 추출
embeddings = embeds(indices_tensor)

embeddings

tensor([[ 1.3983, -0.6694, -0.3386, -0.3865,  0.0711],
        [-0.1304, -0.7972, -0.8293, -0.4809,  0.8129]],
       grad_fn=<EmbeddingBackward0>)

Usually, we define the embedding layer as part of our model, which you will see in the later sections of our notebook.

### 학습 방식: 한 문장씩이 아닌 "배치 단위"로 학습하기

지금까지는 모델 학습을 **"한 문장씩" 처리**하는 방식으로 진행했다.  
하지만 실제 딥러닝에서는 보통 다음의 세 가지 방식 중 하나를 선택한다.

---

### 1. 전체 데이터를 한꺼번에 처리 (Batch Gradient Descent)

- **모든 데이터를 사용해 한 번에 평균 손실(loss)**을 구하고 파라미터 업데이트
- **장점**: 가장 **안정적인 경사(gradient)** 계산 가능  
- **단점**: 데이터가 많을 경우 **속도가 느리고, GPU 메모리 초과** 가능성 있음

---

### 2. 데이터 하나씩 처리 (Stochastic Gradient Descent, SGD)

- **한 문장, 한 예제마다** 파라미터를 바로 업데이트  
- **장점**: 빠르게 학습 시작 가능  
- **단점**: **계산이 불안정**, 손실이 크게 출렁일 수 있음

---

### 3. 적절한 해결책: 미니배치 (Mini-batch Gradient Descent)

- 여러 개의 예제를 묶어 **"배치(batch)" 단위로 학습**
- **장점**:
  - 전체보다는 **빠르고**
  - 개별보다 **안정적인** 경사 계산 가능  
- **PyTorch**에서는 `DataLoader`라는 도구를 통해  
  **미니배치 학습을 쉽게 구현**할 수 있음

---

💡 결론: **속도와 안정성의 균형**을 위해 대부분의 딥러닝 모델은 **Mini-batch 방식**을 사용한다.


In [ ]:
from torch.utils.data import DataLoader
from functools import partial

def custom_collate_fn(batch, window_size, word_to_ix):
    # batch는 [(문장1, 라벨1), (문장2, 라벨2), ...] 형태
    # 문장(x)과 라벨(y)을 각각 분리
    x, y = zip(*batch)

    # 문장 앞뒤에 <pad> 토큰을 window_size만큼 붙이는 함수
    # → 윈도우 기반 문맥 정보를 만들기 위한 전처리
    def pad_window(sentence, window_size, pad_token="<pad>"):
        window = [pad_token] * window_size
        return window + sentence + window

    # 각 문장에 window padding 적용
    x = [pad_window(s, window_size=window_size) for s in x]

    # 단어를 정수 인덱스로 변환하는 함수
    # vocabulary에 없는 단어는 <unk>로 처리
    def convert_tokens_to_indices(sentence, word_to_ix):
        return [word_to_ix.get(token, word_to_ix["<unk>"]) for token in sentence]

    # 각 문장을 단어 인덱스 시퀀스로 변환
    x = [convert_tokens_to_indices(s, word_to_ix) for s in x]


    # 배치 내 문장 길이를 동일하게 맞추기 위해 패딩 수행
    # batch_first=True 이므로 결과 텐서 shape은 (batch_size, seq_len)
    pad_token_ix = word_to_ix["<pad>"]
    x = [torch.LongTensor(x_i) for x_i in x]
    x_padded = nn.utils.rnn.pad_sequence(
        x,
        batch_first=True,
        padding_value=pad_token_ix
    )

    # 각 문장의 실제 라벨 길이 저장
    # → 나중에 패딩을 제외한 실제 단어 길이를 알기 위해 사용
    lengths = [len(label) for label in y]
    lengths = torch.LongTensor(lengths)

    # 라벨도 문장 길이에 맞게 패딩
    # pad 위치는 학습 대상이 아니므로 0으로 채움
    y = [torch.LongTensor(y_i) for y_i in y]
    y_padded = nn.utils.rnn.pad_sequence(
        y,
        batch_first=True,
        padding_value=0
    )

    # 반환값:
    # x_padded  : 패딩된 입력 문장 인덱스 텐서
    # y_padded  : 패딩된 라벨 텐서
    # lengths   : 각 문장의 실제 길이
    return x_padded, y_padded, lengths

이 함수는 길어 보이지만 꼭 그럴 필요는 없다. 별도 함수 선언와 커멘트를 제거한 아래의 버전을 보자.

In [ ]:
def _custom_collate_fn(batch, window_size, word_to_ix):
    # batch를 입력 문장(x)과 라벨(y)로 분리
    x, y = zip(*batch)

    # 문장 앞뒤에 window padding 적용 후
    # 단어를 정수 인덱스로 변환
    x = [pad_window(s, window_size=window_size) for s in x]
    x = [convert_tokens_to_indices(s, word_to_ix) for s in x]

    # 배치 내 문장 길이를 동일하게 맞추기 위해 패딩 수행
    pad_token_ix = word_to_ix["<pad>"]
    x = [torch.LongTensor(x_i) for x_i in x]
    x_padded = nn.utils.rnn.pad_sequence(
        x,
        batch_first=True,
        padding_value=pad_token_ix
    )

    # 각 문장의 실제 라벨 길이 기록
    lengths = [len(label) for label in y]
    lengths = torch.LongTensor(lengths)

    # 라벨도 동일한 길이가 되도록 패딩 (pad 위치는 0)
    y = [torch.LongTensor(y_i) for y_i in y]
    y_padded = nn.utils.rnn.pad_sequence(
        y,
        batch_first=True,
        padding_value=0
    )

    return x_padded, y_padded, lengths

### `torch.utils.data.DataLoader`란?

`DataLoader`는 PyTorch에서 **데이터를 배치 단위로 자동으로 불러오고**,  
**모델에 넣기 편하게 구성**해주는 유틸리티 클래스이다.

---

### 주요 기능

- **배치 단위(batch-wise)**로 데이터 불러오기  
- **셔플(shuffling)** 기능 지원 → 데이터 순서를 무작위로 섞어 일반화 향상  
- **병렬 처리(num_workers)**로 데이터 로딩 속도 향상 가능  
- **이터레이터 형태**로 동작 → `for batch in dataloader:`처럼 사용 가능

---

### 왜 필요한가?

학습 시 데이터를 직접 한 줄씩 불러오는 것은 번거롭고 비효율적이다.  
`DataLoader`는 이를 자동화하여 **모델 학습을 더 효율적이고 안정적으로** 만들어 준다

---

### 요약

> ✅ `DataLoader`는 PyTorch에서 **모델 학습을 위한 데이터 관리와 배치 처리**를 도와주는 핵심 도구이다.


In [ ]:
# DataLoader에 전달할 데이터 구성 (문장, 라벨 쌍)
data = list(zip(train_sentences, train_labels))

# 배치 크기 설정 (한 번에 모델에 넣을 샘플 수)
batch_size = 2

# 학습 시 데이터 순서를 섞을지 여부
shuffle = True

# 문장 앞뒤에 붙일 window padding 크기
window_size = 2

# collate 함수에 window_size와 word_to_ix를 미리 전달하기 위해 partial 사용
collate_fn = partial(custom_collate_fn,
                     window_size=window_size,
                     word_to_ix=word_to_ix)

# DataLoader 객체 생성
# → 데이터를 배치 단위로 나누고, collate_fn으로 전처리 수행
loader = DataLoader(data,
                    batch_size=batch_size,
                    shuffle=shuffle,
                    collate_fn=collate_fn)

# DataLoader가 생성한 배치를 한 번 순회하며 확인
counter = 0
for batched_x, batched_y, batched_lengths in loader:
    print(f"Iteration {counter}")
    print("Batched Input:")
    print(batched_x)
    print("Batched Labels:")
    print(batched_y)
    print("Batched Lengths:")
    print(batched_lengths)
    print("")
    counter += 1

Iteration 0
Batched Input:
tensor([[ 0,  0, 10, 13, 11, 17,  0,  0,  0],
        [ 0,  0, 22,  2,  6, 20, 15,  0,  0]])
Batched Labels:
tensor([[0, 0, 0, 1, 0],
        [0, 0, 0, 0, 1]])
Batched Lengths:
tensor([4, 5])

Iteration 1
Batched Input:
tensor([[ 0,  0,  9,  7,  8, 18,  0,  0,  0],
        [ 0,  0, 19, 16, 12,  8,  4,  0,  0]])
Batched Labels:
tensor([[0, 0, 0, 1, 0],
        [0, 0, 0, 0, 1]])
Batched Lengths:
tensor([4, 5])

Iteration 2
Batched Input:
tensor([[ 0,  0, 19,  5, 14, 21, 12,  3,  0,  0]])
Batched Labels:
tensor([[0, 0, 0, 1, 0, 1]])
Batched Lengths:
tensor([6])



### 모델 입력: 윈도우 기반 분류기를 위한 전처리 완료

앞서 만든 배치 입력 텐서들은 이제 **모델에 전달될 준비**가 되었다.  
우리가 만들 모델은 **윈도우 기반 분류기**(Window-based Classifier)이다.

---

### 데이터 구조와 예측 방식

- 현재 입력된 텐서들은 **한 문장의 모든 단어가 하나의 데이터 포인트로 묶인 형태**이다.
- 모델은 이 데이터를 입력으로 받아 **각 단어를 중심으로 윈도우**를 만들고,
- **중심 단어가 LOCATION인지 아닌지를 예측**해야 한다.
- 예측 결과는 **전체 문장에 대한 위치 정보** (**예: [0, 0, 0, 1]**)로 반환된다.

---

### 윈도우 단위로 데이터를 미리 자를 수도 있지만...

- 데이터를 **사전에 윈도우 단위로 잘라 모델에 넣는 방식도 가능**하지만,  
- 이런 윈도우 생성 과정을 **모델 내부에서 처리**할 수도 있다.  
- 이 방식은 **모델을 더 일반적이고 유연하게** 만들어 준다.

---

### 윈도우 크기와 예측 횟수

- 윈도우 크기를 `N`이라고 하면, 매번 **2N + 1개의 단어**를 보고 예측을 수행
- 예시: 입력로 9개의 단어, 윈도우 크기 2 → 총 **5개의 예측 수행**

---

### 윈도우 생성: `unfold()` 메서드 활용

PyTorch에서는 **효율적으로 윈도우를 생성**할 수 있는 메서드가 존재한다:

```python
tensor.unfold(dimension, size, step)


In [ ]:
# 패딩된 입력 텐서 확인
print(f"Original Tensor: ")
print(batched_x)
print("")

# unfold(dim, size, step)
# → 시퀀스에서 슬라이딩 윈도우 형태의 부분 텐서들을 생성
# dim=1 : 시퀀스 길이 방향으로 윈도우 생성
# size = window_size*2 + 1 : 중심 단어 기준 좌우 문맥 포함한 윈도우 크기
# step=1 : 한 칸씩 이동하며 윈도우 생성
chunk = batched_x.unfold(1, window_size*2 + 1, 1)

print(f"Windows: ")
print(chunk)

Original Tensor: 
tensor([[ 0,  0, 19,  5, 14, 21, 12,  3,  0,  0]])

Windows: 
tensor([[[ 0,  0, 19,  5, 14],
         [ 0, 19,  5, 14, 21],
         [19,  5, 14, 21, 12],
         [ 5, 14, 21, 12,  3],
         [14, 21, 12,  3,  0],
         [21, 12,  3,  0,  0]]])


### 모델

이제 데이터 준비가 끝났으므로 모델을 구축할 수 있음  
앞에서 배운 사용자 정의 `nn.Module` 클래스 작성 방법을 바탕으로,  
지금까지 학습한 내용을 종합하여 모델을 구현할 것.

In [ ]:
class WordWindowClassifier(nn.Module):

    def __init__(self, hyperparameters, vocab_size, pad_ix=0):
        super(WordWindowClassifier, self).__init__()

        # 하이퍼파라미터 저장
        self.window_size = hyperparameters["window_size"]
        self.embed_dim = hyperparameters["embed_dim"]
        self.hidden_dim = hyperparameters["hidden_dim"]
        self.freeze_embeddings = hyperparameters["freeze_embeddings"]

        # 임베딩 층
        # 입력: 단어 인덱스
        # 출력: 해당 단어의 임베딩 벡터
        # padding_idx=pad_ix : pad 토큰은 임베딩 학습에서 제외됨
        self.embeds = nn.Embedding(vocab_size, self.embed_dim, padding_idx=pad_ix)

        # freeze_embeddings=True이면 임베딩 가중치를 학습하지 않도록 고정
        if self.freeze_embeddings:
            self.embeds.weight.requires_grad = False

        # 은닉층
        # 하나의 윈도우 크기 = 중심 단어 1개 + 좌우 문맥
        full_window_size = 2 * self.window_size + 1

        # 윈도우 내 단어 임베딩들을 펼친 뒤 선형변환 + Tanh 적용
        self.hidden_layer = nn.Sequential(
            nn.Linear(full_window_size * self.embed_dim, self.hidden_dim),
            nn.Tanh()
        )

        # 출력층
        # 각 단어가 특정 클래스(예: 지명)인지에 대한 점수 1개 출력
        self.output_layer = nn.Linear(self.hidden_dim, 1)

        # 이진 분류 확률로 변환
        self.probabilities = nn.Sigmoid()

    def forward(self, inputs):
        """
        B := batch_size
        L := window-padding이 적용된 문장 길이
        D := 임베딩 차원
        S := window_size
        H := 은닉층 차원

        inputs: (B, L) 크기의 단어 인덱스 텐서
        """
        B, L = inputs.size()

        # 입력 문장에서 각 단어 기준의 슬라이딩 윈도우 생성
        # 입력  : (B, L)
        # 출력  : (B, L~, 2S+1)
        token_windows = inputs.unfold(1, 2 * self.window_size + 1, 1)
        _, adjusted_length, _ = token_windows.size()

        # 내부 shape 확인
        assert token_windows.size() == (B, adjusted_length, 2 * self.window_size + 1)

        # 각 윈도우의 단어 인덱스를 임베딩 벡터로 변환
        # 입력  : (B, L~, 2S+1)
        # 출력  : (B, L~, 2S+1, D)
        embedded_windows = self.embeds(token_windows)

        # 선형층 입력을 위해 윈도우 임베딩을 1차원으로 펼침
        # 입력  : (B, L~, 2S+1, D)
        # 출력  : (B, L~, (2S+1)*D)
        embedded_windows = embedded_windows.view(B, adjusted_length, -1)

        # 은닉층 통과
        # 입력  : (B, L~, (2S+1)*D)
        # 출력  : (B, L~, H)
        layer_1 = self.hidden_layer(embedded_windows)

        # 출력층 통과
        # 입력  : (B, L~, H)
        # 출력  : (B, L~, 1)
        output = self.output_layer(layer_1)

        # Sigmoid를 적용해 각 위치의 이진 분류 확률로 변환
        output = self.probabilities(output)

        # 마지막 차원 제거
        # 출력 shape: (B, L~)
        output = output.view(B, -1)

        return output

Embedding layer: 단어 인덱스를 임베딩 벡터로 변환

Window: 각 단어 주변 문맥을 함께 보는 입력 단위

Hidden layer: 윈도우 정보를 바탕으로 특징 추출

Output layer: 각 단어가 목표 클래스인지 점수 출력

Sigmoid: 점수를 0~1 사이 확률로 변환

### 임베딩 벡터 평탄화(Flatten)란?
여러 개의 임베딩 벡터를 한 개의 긴 벡터로 이어 붙이는 것

### Training

이제 모든 준비가 완료되었으므로 전체 과정을 하나로 연결해보자
먼저 데이터를 준비하고 모델을 초기화한다.  
그 다음 옵티마이저를 설정하고 손실 함수를 정의한다.  

이번에는 이전처럼 미리 정의된 손실 함수를 사용하는 대신,  
직접 손실 함수를 구현해볼 것이다.

In [ ]:
# 학습 데이터 준비 (문장, 라벨 쌍)
data = list(zip(train_sentences, train_labels))

# DataLoader 설정값
batch_size = 2
shuffle = True
window_size = 2

# collate 함수에 필요한 추가 인자를 미리 고정
collate_fn = partial(custom_collate_fn,
                     window_size=window_size,
                     word_to_ix=word_to_ix)

# DataLoader 생성
# → 데이터를 배치 단위로 불러오고, collate_fn으로 전처리 수행
loader = DataLoader(data,
                    batch_size=batch_size,
                    shuffle=shuffle,
                    collate_fn=collate_fn)

# 모델 초기화
# 하이퍼파라미터를 딕셔너리로 정리하면 관리가 편리함
model_hyperparameters = {
    "batch_size": 4,
    "window_size": 2,
    "embed_dim": 25,
    "hidden_dim": 25,
    "freeze_embeddings": False,
}

vocab_size = len(word_to_ix)
model = WordWindowClassifier(model_hyperparameters, vocab_size)

# 옵티마이저 정의
# SGD를 사용하여 모델 파라미터를 업데이트
learning_rate = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# 손실 함수 정의
# 이진 분류 문제이므로 Binary Cross Entropy Loss 사용
def loss_function(batch_outputs, batch_labels, batch_lengths):
    # 배치 전체에 대한 BCE 손실 계산
    bceloss = nn.BCELoss()
    loss = bceloss(batch_outputs, batch_labels.float())

    # 각 문장의 실제 길이 합으로 나누어 손실 크기 보정
    # → 패딩의 영향을 줄이고 실제 단어 수 기준으로 정규화
    loss = loss / batch_lengths.sum().float()

    return loss

DataLoader: 데이터를 배치 단위로 공급

optimizer: gradient를 이용해 가중치 업데이트

BCELoss: 이진 분류에서 예측 확률과 정답 차이를 계산

batch_lengths: 실제 단어 수 기준으로 손실을 보정하기 위해 사용

-----------------

이전 예제와는 달리, 이번에는 매 epoch마다 전체 학습 데이터를 한 번에 모델에 입력하지 않는다.  
대신 데이터를 **배치(batch) 단위로 나누어 학습**을 진행한다.  

따라서 각 학습 epoch마다  
여러 개의 배치를 순차적으로 반복(iteration)하면서 모델을 학습하게 된다.

In [ ]:
# 한 epoch 동안 모델을 학습시키는 함수
def train_epoch(loss_function, optimizer, model, loader):

    total_loss = 0

    # DataLoader가 생성한 배치들을 순회
    for batch_inputs, batch_labels, batch_lengths in loader:

        # 이전 step에서 계산된 gradient 초기화
        optimizer.zero_grad()

        # 순전파(forward pass) 수행 → 모델 예측값 계산
        outputs = model.forward(batch_inputs)

        # 배치 손실 계산
        loss = loss_function(outputs, batch_labels, batch_lengths)

        # 역전파(backpropagation) → gradient 계산
        loss.backward()

        # 옵티마이저를 통해 가중치 업데이트
        optimizer.step()

        total_loss += loss.item()

    return total_loss


# 전체 학습 루프
# 여러 epoch 동안 train_epoch를 반복 실행
def train(loss_function, optimizer, model, loader, num_epochs=10000):

    for epoch in range(num_epochs):
        epoch_loss = train_epoch(loss_function, optimizer, model, loader)

        # 일정 epoch마다 손실 출력
        if epoch % 100 == 0:
            print(epoch_loss)

epoch: 전체 학습 데이터를 한 번 학습하는 과정

batch 학습: 데이터를 여러 묶음으로 나누어 반복 학습

forward → loss → backward → update
→ 딥러닝 학습의 기본 흐름

optimizer.step()
→ gradient를 이용해 모델 파라미터 수정

------

훈련 시작 !



In [ ]:
num_epochs = 1000
train(loss_function, optimizer, model, loader, num_epochs=num_epochs)

0.2606007717549801
0.20252665132284164
0.16072198003530502
0.1271534599363804
0.120423574000597
0.08107922039926052
0.0636079590767622
0.04935570992529392
0.0404148930683732
0.031674101017415524


### Prediction

이제 우리 모델이 얼마나 잘 예측하는지 확인해보자.  
먼저 테스트 데이터를 생성하는 것부터 시작한다.

In [ ]:
# 테스트 문장 생성
test_corpus = ["She comes from Paris"]

# 소문자 변환 후 공백 기준 토큰화
test_sentences = [s.lower().split() for s in test_corpus]

# 테스트 정답 라벨 (각 단어별 이진 라벨)
test_labels = [[0, 0, 0, 1]]

# 테스트 데이터 구성 (문장, 라벨 쌍)
test_data = list(zip(test_sentences, test_labels))

# 테스트용 DataLoader 설정
batch_size = 1
shuffle = False
window_size = 2

# 학습 때와 동일한 전처리를 위해 같은 collate 함수 사용
collate_fn = partial(custom_collate_fn,
                     window_size=2,
                     word_to_ix=word_to_ix)

# 테스트용 DataLoader 생성
test_loader = torch.utils.data.DataLoader(
    test_data,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn
)

테스트 데이터를 순회하면서 모델이 얼마나 잘 예측하는지 확인해보자.



In [ ]:
# 테스트 데이터 배치를 순회하며 모델 예측 확인
for test_instance, labels, _ in test_loader:

    # 순전파 수행 → 각 단어에 대한 예측 확률 계산
    outputs = model.forward(test_instance)

    # 실제 라벨과 모델 예측값 출력
    print(labels)
    print(outputs)

tensor([[0, 0, 0, 1]])
tensor([[0.0541, 0.0426, 0.1068, 0.8918]], grad_fn=<ViewBackward0>)
